### ABOUT Stack MVP

Meeting Prep Document ExxonMobil - Discovery call- 3/12, 10am 
Derek Naji - Dir of Architecture 

Close Date: 12/30/2026 
Deal Size: $250k ARR 
Grade: Pipeline 
Notes :  (link  to Notes) 

3 emails in last 7 days (link to emails) 
Product pricing page visited twice last week 
Contact downloaded API Management white paper 

ExxonMobil CIO said in an interview the company intends to upgrade interfacing and data 
exchange technology to improve operations with all vendors and partners. Part of strategy to 
transition to hybrid environment (on-prem, cloud, SaaS) 

ExxonMobil has 2 open jobs for API admins

Stack Meeting Prep MVP feature.

HubSpot: jbeard4@comcast.net/Axxis1000!Hi
https://app-na2.hubspot.com/global-home/245065726

1) login to Stack (login page)
2) go to 'main' page for navigation on which the 
3) Stack AI agent asks: “How can I help you today?”
4) User selects 'Meeting Prep'
5) Stack redirects user to page with his meeting calendar for the week, and asks: “What meeting(s) can I help you prepare for?”
NOTE: In the future, prefer to use API to Outlook, but MVP will use Hubspot Activities section for 'meetings'. 
  Companies – select company (ExxonMobil in this case) – Activities. 
  Mockup has four meetings on calendar: ExxonMobil, Nissan NA, AT&T and Sabre
1) User selects ExxonMobil meeting. Stack returns page with two columns:
  Left side column: Selected Meeting Data
  Right side column: Data Retrieval Flow (displays progress of his/her request- Stack checks off each data category as it locates and extracts it)

The Stack AI agent narrates the data on the Right-hand frame: “I am finding and organizing the data you need”. 
When data is extracted, AI agent asks, “Would you like to see the Meeting Prep document?"
User selects 'Yes'.
Stack generates 'Meeting Prep'

Meeting Prep MVP

1. Meeting info from Outlook calendar (for MVP use Activities-Notes section in each Company)
  --> Company--> meeting info from the calendar
1. CRM basic data

2. Contact title – upper right hand section in company profile. Path = Companies – select company - Contacts

3. Deal/opportunity – section under Contacts. Stack extracts Close Date, Deal Size, Deal Stage. Path = Companies – select company - Deals

4. Notes (recent notes, last 30 days) – section titled Notes. Path = Companies – select company – Notes

5. Recent email communication (last 7 days)
  get link to recent email communications: Companies - select company – Activities – Emails
  get email link (summary? actions?) for the present time

1. Any interaction via Internet from Company or Contact.
   use “dummy data” in Notes section from the Contact who did the 'activity'.
   Path = Companies, select company – select Contact - Activities – Notes
   eg. Derrick Naji at ExxonMobil visited the Pricing page and downloaded a white paper (both relevant actions)

Assume for MVP this data is separate and not integrated; Stack will pull data from applications in the sales tech stack. 


============================
setup Conda env: stack
pip install hubspot-api-client python-dotenv pandas requests
pip install ipykernel jupyterlab


pip install openpyxl xlsxwriter


### LOGGER

In [119]:
import sys
sys.path.insert(0, r"C:\dev\stack")   # so `utils` imports work

from utils.tee_logger import enable_tee

enable_tee(r"C:\dev\stack\utils\output.log")
print("Logger enabled for Jupyter and Bash Terminal.")

# in gitBash, tail the log with command: tail -f utils/output.log  (currently a bash alias called 'logtail')

Logger enabled for Jupyter and Bash Terminal.


### CB0: Imports

In [120]:
# =========================
# CB0 — Imports
# =========================
import os
import html
import requests
import pandas as pd
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from dotenv import load_dotenv
from hubspot import HubSpot

### CB1: CSS

In [121]:
display(HTML("""
<style>
.stack-root {
    width: 100%;
    max-width: 100%;
    font-family: Arial, sans-serif;
    box-sizing: border-box;
}

/* header row */
.stack-header-row {
    background: #b3b3b3;
    padding: 14px 16px;
    margin-bottom: 18px;
    box-sizing: border-box;
    border-radius: 0;
}

.stack-header-title {
    color: white;
    text-align: center;
    font-size: 32px;
    font-weight: 500;
    line-height: 1.2;
}

/* stack agent card */
.stack-agent-card {
    background: #b3b3b3;
    color: white;
    border-radius: 18px;
    padding: 18px 16px;
    min-height: 160px;
    box-sizing: border-box;
}

.stack-agent-title {
    font-size: 20px;
    font-weight: 600;
    text-decoration: underline;
    margin-bottom: 20px;
}

.stack-agent-text {
    font-size: 18px;
    line-height: 1.4;
    white-space: pre-wrap;
}

/* shared panels */
.stack-panel {
    border: 1px solid #d8d8d8;
    border-radius: 10px;
    background: #f7f7f7;
    padding: 12px;
    box-sizing: border-box;
}

.stack-panel-title {
    font-size: 16px;
    font-weight: 600;
    margin-bottom: 10px;
}

.stack-output-placeholder {
    color: #666;
    font-size: 13px;
}

/* main action buttons: active = light green */
.widget-button.stack-main-btn > button {
    background: #b7e4c7 !important;
    color: #1f3d2b !important;
    border: 1px solid #95d5b2 !important;
    border-radius: 18px !important;
    font-size: 18px !important;
    font-weight: 500 !important;
    height: 78px !important;
    width: 100% !important;
}

.widget-button.stack-main-btn > button:hover {
    filter: brightness(0.97);
}

.widget-button.stack-main-btn > button:disabled {
    background: #efefef !important;
    color: #777 !important;
    border: 1px solid #d6d6d6 !important;
    opacity: 1 !important;
}

/* login button */
.widget-button.stack-login-btn > button {
    background: white !important;
    color: #444 !important;
    border: 1px solid #d0d0d0 !important;
    border-radius: 8px !important;
    font-size: 13px !important;
    font-weight: 600 !important;
    height: 34px !important;
    width: 100px !important;
    min-width: 100px !important;
    padding: 0 10px !important;
}

.widget-button.stack-login-btn > button:hover {
    filter: brightness(0.97);
}

.widget-button.stack-login-btn > button:disabled {
    background: #efefef !important;
    color: #666 !important;
    border: 1px solid #cfcfcf !important;
    opacity: 1 !important;
}

/* logout button */
.widget-button.jupyter-widgets.widget-button.button_style_warning > button {
    border-radius: 8px !important;
    font-size: 13px !important;
    font-weight: 600 !important;
}

/* make output/debug tables easier to read */
.stack-panel table {
    font-size: 14px;
}

.stack-panel .dataframe {
    margin-bottom: 10px;
}
</style>
"""))

### CB2: State / Config / Flags

In [122]:
# =========================
# CB2 — State / Config / Flags
# =========================

if "STACK_UI_STATE" not in globals() or not isinstance(STACK_UI_STATE, dict):
    STACK_UI_STATE = {}

if "HS_CONTEXT" not in globals() or not isinstance(HS_CONTEXT, dict):
    HS_CONTEXT = {}

if "api" not in globals():
    api = None

pd.set_option("display.max_colwidth", None)

OWNER_ID_COL = "hubspot_owner_id"
DEBUG_IDENTITY = False
DEBUG_USERID = False

STACK_UI_DEFAULTS = {
    "agent_text": "How can I help you today?",
    "last_action": None,
    "logged_in": False,
    "hubspot_login_info": {},
    "token_user_email": "",
    "token_user_id": "",
    "portal_id": "",
    "portal_name": "",
    "meetings_df": None,
    "companies_df": None,
    "contacts_df": None,
    "companies_display_df": None,
    "my_owned_companies_df": None,
}

for _k, _v in STACK_UI_DEFAULTS.items():
    STACK_UI_STATE.setdefault(_k, _v)


    

print("CB2 done: State / Config / Flags")

CB2 done: State / Config / Flags


### CB3: General Helpers

In [123]:
# =========================
# CB3 — Pure / General Helpers
# =========================

def _esc(x) -> str:
    return html.escape("" if x is None else str(x))


def _clean_str(x) -> str:
    s = "" if x is None else str(x).strip()
    return "" if s in {"", "nan", "None"} else s


def _require_login() -> None:
    if api is None or not STACK_UI_STATE.get("logged_in"):
        raise RuntimeError("Not logged in. Click 'Log In' first.")


def sync_action_buttons():
    active = bool(STACK_UI_STATE.get("logged_in"))

    btns = [
        opp_insights_btn,
        meeting_prep_btn,
        my_companies_btn,
        opp_search_btn,
    ]

    for btn in btns:
        btn.disabled = not active


def _ensure_columns(df: pd.DataFrame | None, out_cols: list[str]) -> pd.DataFrame:
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return pd.DataFrame(columns=out_cols)

    df = df.copy()
    for col in out_cols:
        if col not in df.columns:
            df[col] = ""

    return df[out_cols].copy()


def set_agent_text(text: str) -> None:
    STACK_UI_STATE["agent_text"] = str(text)

    if "agent_card" in globals():
        agent_card.value = (
            "<div class='stack-agent-card'>"
            "  <div class='stack-agent-title'>Stack Agent</div>"
            f"  <div class='stack-agent-text'>{_esc(text)}</div>"
            "</div>"
        )


def show_output(title: str, content=None) -> None:
    if "output_area" not in globals():
        return

    with output_area:
        clear_output()
        display(HTML(f"<h4 style='margin:0 0 10px 0;'>{_esc(title)}</h4>"))

        if content is None:
            display(HTML("<div class='stack-output-placeholder'>No output returned.</div>"))
        elif isinstance(content, pd.DataFrame):
            display(content)
        elif isinstance(content, str):
            display(HTML(f"<div style='white-space:pre-wrap;'>{_esc(content)}</div>"))
        else:
            display(content)


def build_login_display_df(login_info: dict) -> pd.DataFrame:
    login_info = dict(login_info or {})
    return pd.DataFrame([{
        "portal_id": login_info.get("portal_id", ""),
        "portal_name": login_info.get("portal_name", ""),
        "portal_timezone": login_info.get("portal_timezone", ""),
        "portal_currency": login_info.get("portal_currency", ""),
        "token_user_email": login_info.get("token_user_email", ""),
        "token_user_id": login_info.get("token_user_id", ""),
        "token_hub_id": login_info.get("token_hub_id", ""),
        "token_app_id": login_info.get("token_app_id", ""),
    }])


def build_connected_agent_text(login_info: dict) -> str:
    login_info = dict(login_info or {})

    user_id = _clean_str(login_info.get("token_user_id", ""))
    hub_id = _clean_str(login_info.get("token_hub_id", ""))
    app_id = _clean_str(login_info.get("token_app_id", ""))
    email = _clean_str(login_info.get("token_user_email", ""))
    portal_name = _clean_str(login_info.get("portal_name", ""))
    portal_id = _clean_str(login_info.get("portal_id", ""))

    who = (
        email
        or (f"userId {user_id}" if user_id else "")
        or portal_name
        or (f"Portal {portal_id}" if portal_id else "HubSpot")
    )

    lines = [f"Connected as {who}."]
    if user_id:
        lines.append(f"userId = {user_id}")
    if hub_id:
        lines.append(f"hubId = {hub_id}")
    if app_id:
        lines.append(f"appId = {app_id}")

    return "\n".join(lines)


def get_company_homepage_url(row) -> str:
    website = _clean_str(row.get("website", ""))
    domain = _clean_str(row.get("domain", ""))

    url = website or domain
    if not url:
        return ""

    if not url.startswith(("http://", "https://")):
        url = f"https://{url}"

    return url


def _payload_preview_df(payload, max_rows: int = 10) -> pd.DataFrame:
    if isinstance(payload, dict):
        if isinstance(payload.get("results"), list):
            return pd.DataFrame(payload["results"][:max_rows])

        flat = {}
        for k, v in payload.items():
            flat[k] = str(v) if isinstance(v, (dict, list)) else v
        return pd.DataFrame([flat])

    if isinstance(payload, list):
        return pd.DataFrame(payload[:max_rows])

    return pd.DataFrame([{"value": str(payload)}])

print("CB3 done: Pure / General Helpers")

CB3 done: Pure / General Helpers


### CB4: Login / Probes / DEBUG Helpers

In [124]:
# =========================
# CB4 — Login / Probes / DEBUG Helpers
# =========================

def _get_hs_token() -> str:
    load_dotenv()
    token = (os.getenv("HS_TOKEN") or "").strip()
    assert token, "HS_TOKEN not found. Check .env and notebook working directory."
    return token


def _get_hs_headers(token: str) -> dict:
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }


def run_hubspot_login() -> dict:
    global api, HS_CONTEXT

    token = _get_hs_token()
    headers = _get_hs_headers(token)

    api = HubSpot(access_token=token)

    acct_resp = requests.get(
        "https://api.hubapi.com/account-info/2026-03/details",
        headers=headers,
        timeout=30,
    )
    acct_resp.raise_for_status()
    acct = acct_resp.json()

    tokn_resp = requests.post(
        "https://api.hubapi.com/oauth/v2/private-apps/get/access-token-info",
        headers=headers,
        json={"tokenKey": token},
        timeout=30,
    )
    tokn_resp.raise_for_status()
    tokn = tokn_resp.json()

    HS_CONTEXT = {
        "portal_id": _clean_str(acct.get("portalId", "")),
        "portal_name": _clean_str(acct.get("accountName", "")),
        "portal_timezone": _clean_str(acct.get("timeZone", "")),
        "portal_currency": _clean_str(acct.get("companyCurrency", "")),
        "token_user_id": _clean_str(tokn.get("userId", "")),
        "token_hub_id": _clean_str(tokn.get("hubId", "")),
        "token_app_id": _clean_str(tokn.get("appId", "")),
        "token_scopes": tokn.get("scopes", []) or [],
        "token_user_email": _clean_str(((tokn.get("user") or {}).get("email", ""))),
    }

    STACK_UI_STATE["logged_in"] = True
    STACK_UI_STATE["hubspot_login_info"] = dict(HS_CONTEXT)
    STACK_UI_STATE["token_user_email"] = HS_CONTEXT["token_user_email"]
    STACK_UI_STATE["token_user_id"] = HS_CONTEXT["token_user_id"]
    STACK_UI_STATE["portal_id"] = HS_CONTEXT["portal_id"]
    STACK_UI_STATE["portal_name"] = HS_CONTEXT["portal_name"]

    return HS_CONTEXT


def reset_hubspot_login_state() -> None:
    global api, HS_CONTEXT

    api = None
    HS_CONTEXT = {}

    STACK_UI_STATE["logged_in"] = False
    STACK_UI_STATE["hubspot_login_info"] = {}
    STACK_UI_STATE["token_user_email"] = ""
    STACK_UI_STATE["token_user_id"] = ""
    STACK_UI_STATE["portal_id"] = ""
    STACK_UI_STATE["portal_name"] = ""
    STACK_UI_STATE["last_action"] = "hubspot_logout"
    STACK_UI_STATE["meetings_df"] = None
    STACK_UI_STATE["companies_df"] = None
    STACK_UI_STATE["contacts_df"] = None
    STACK_UI_STATE["companies_display_df"] = None
    STACK_UI_STATE["my_owned_companies_df"] = None


def _probe_endpoint(label: str, method: str, url: str, headers: dict, json_body=None) -> dict:
    try:
        if method.upper() == "GET":
            resp = requests.get(url, headers=headers, timeout=30)
        elif method.upper() == "POST":
            resp = requests.post(url, headers=headers, json=json_body, timeout=30)
        else:
            raise ValueError(f"Unsupported method: {method}")

        try:
            payload = resp.json()
        except Exception:
            payload = {"raw_text": resp.text[:1000]}

        return {
            "label": label,
            "method": method.upper(),
            "url": url,
            "status_code": resp.status_code,
            "ok": resp.ok,
            "payload": payload,
        }

    except Exception as e:
        return {
            "label": label,
            "method": method.upper(),
            "url": url,
            "status_code": None,
            "ok": False,
            "payload": {"error": str(e)},
        }


def run_login_probes() -> list[dict]:
    token = _get_hs_token()
    headers = _get_hs_headers(token)

    return [
        _probe_endpoint(
            "Account Details",
            "GET",
            "https://api.hubapi.com/account-info/2026-03/details",
            headers,
        ),
        _probe_endpoint(
            "Access Token Info",
            "POST",
            "https://api.hubapi.com/oauth/v2/private-apps/get/access-token-info",
            headers,
            json_body={"tokenKey": token},
        ),
        _probe_endpoint(
            "Owners",
            "GET",
            "https://api.hubapi.com/crm/owners/2026-03",
            headers,
        ),
        _probe_endpoint(
            "Users",
            "GET",
            "https://api.hubapi.com/settings/users/2026-03",
            headers,
        ),
    ]


def _probe_payload_results(payload) -> list[dict]:
    if isinstance(payload, dict) and isinstance(payload.get("results"), list):
        return payload.get("results") or []
    return []


def run_userid_debug_probes(
    login_info: dict | None = None,
    companies_df: pd.DataFrame | None = None,
    max_owner_ids: int = 3,
) -> dict:
    """
    Probe a few user/owner-related HubSpot endpoints so we can see
    what works for the current token.
    """
    login_info = dict(login_info or STACK_UI_STATE.get("hubspot_login_info") or {})

    token = _get_hs_token()
    headers = _get_hs_headers(token)

    token_user_id = _clean_str(login_info.get("token_user_id", ""))
    token_hub_id = _clean_str(login_info.get("token_hub_id", ""))
    token_app_id = _clean_str(login_info.get("token_app_id", ""))

    probes = [
        _probe_endpoint(
            "Account Details",
            "GET",
            "https://api.hubapi.com/account-info/2026-03/details",
            headers,
        ),
        _probe_endpoint(
            "Access Token Info",
            "POST",
            "https://api.hubapi.com/oauth/v2/private-apps/get/access-token-info",
            headers,
            json_body={"tokenKey": token},
        ),
        _probe_endpoint(
            "Owners List",
            "GET",
            "https://api.hubapi.com/crm/owners/2026-03",
            headers,
        ),
        _probe_endpoint(
            "Users List",
            "GET",
            "https://api.hubapi.com/settings/users/2026-03",
            headers,
        ),
    ]

    if token_user_id:
        probes.append(
            _probe_endpoint(
                f"User By Token userId {token_user_id}",
                "GET",
                f"https://api.hubapi.com/settings/users/2026-03/{token_user_id}",
                headers,
            )
        )
        probes.append(
            _probe_endpoint(
                f"Owner By Token userId As OwnerId {token_user_id}",
                "GET",
                f"https://api.hubapi.com/crm/owners/2026-03/{token_user_id}",
                headers,
            )
        )

    if companies_df is None:
        companies_df = STACK_UI_STATE.get("companies_df")
        if companies_df is None or not isinstance(companies_df, pd.DataFrame) or companies_df.empty:
            try:
                companies_df = get_my_companies()
            except Exception:
                companies_df = pd.DataFrame()

    if not isinstance(companies_df, pd.DataFrame):
        companies_df = pd.DataFrame()

    company_owner_ids = []
    if not companies_df.empty and "hubspot_owner_id" in companies_df.columns:
        company_owner_ids = sorted({
            _clean_str(x)
            for x in companies_df["hubspot_owner_id"].tolist()
            if _clean_str(x)
        })

    for owner_id in company_owner_ids[:max_owner_ids]:
        probes.append(
            _probe_endpoint(
                f"Owner By Company Owner ID {owner_id}",
                "GET",
                f"https://api.hubapi.com/crm/owners/2026-03/{owner_id}",
                headers,
            )
        )

    login_df = pd.DataFrame([{
        "token_user_id": token_user_id,
        "token_hub_id": token_hub_id,
        "token_app_id": token_app_id,
        "token_user_email": _clean_str(login_info.get("token_user_email", "")),
        "portal_id": _clean_str(login_info.get("portal_id", "")),
        "portal_name": _clean_str(login_info.get("portal_name", "")),
    }])

    owner_ids_df = pd.DataFrame({"company_owner_id": company_owner_ids})

    probe_summary_df = pd.DataFrame([
        {
            "label": p.get("label", ""),
            "status_code": p.get("status_code", ""),
            "ok": p.get("ok", False),
            "url": p.get("url", ""),
        }
        for p in probes
    ])

    return {
        "login_df": login_df,
        "owner_ids_df": owner_ids_df,
        "probe_summary_df": probe_summary_df,
        "probes": probes,
    }


def render_userid_debug(bundle: dict) -> None:
    if "debug_output_area" not in globals():
        return

    login_df = bundle.get("login_df", pd.DataFrame())
    owner_ids_df = bundle.get("owner_ids_df", pd.DataFrame())
    probe_summary_df = bundle.get("probe_summary_df", pd.DataFrame())
    probes = bundle.get("probes", []) or []

    set_debug_visible(True)

    with debug_output_area:
        clear_output()

        display(HTML("<h4 style='margin:0 0 10px 0;'>DEBUG_USERID</h4>"))

        display(HTML("<div style='margin:0 0 8px 0; font-weight:600;'>Token / Portal Identity</div>"))
        display(login_df)

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Company Owner IDs Found</div>"))
        display(owner_ids_df)

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Probe Summary</div>"))
        display(probe_summary_df)

        for p in probes:
            label = p.get("label", "")
            status_code = p.get("status_code", "")
            ok = p.get("ok", False)
            payload = p.get("payload", {})

            display(HTML(
                f"<div style='margin:14px 0 6px 0; font-weight:600;'>"
                f"{_esc(label)} — status {status_code} — ok={ok}"
                f"</div>"
            ))
            display(_payload_preview_df(payload))


def handle_userid_debug(
    login_info: dict | None = None,
    companies_df: pd.DataFrame | None = None,
) -> None:
    if DEBUG_USERID:
        bundle = run_userid_debug_probes(
            login_info=login_info,
            companies_df=companies_df,
        )
        render_userid_debug(bundle)
    else:
        clear_debug_output()

        

def _build_people_preview(results: list[dict], max_items: int = 3) -> str:
    parts = []

    for row in (results or [])[:max_items]:
        email = _clean_str(row.get("email", ""))
        first = _clean_str(row.get("firstName", ""))
        last = _clean_str(row.get("lastName", ""))
        user_id = _clean_str(row.get("userId", ""))
        obj_id = _clean_str(row.get("id", ""))

        name = " ".join(x for x in [first, last] if x).strip()

        label = (
            name
            or email
            or (f"userId {user_id}" if user_id else "")
            or (f"id {obj_id}" if obj_id else "")
        )
        if label:
            parts.append(label)

    return ", ".join(parts)


def build_stack_agent_login_text(
    login_info: dict | None = None,
    probes: list[dict] | None = None,
) -> str:
    """
    Build Stack Agent text after login using whatever identity/user/owner
    information is actually available to the current token.
    """
    login_info = dict(login_info or STACK_UI_STATE.get("hubspot_login_info") or {})
    probes = list(probes or [])

    probe_by_label = {str(p.get("label", "")): p for p in probes}

    token_user_email = _clean_str(login_info.get("token_user_email", ""))
    token_user_id = _clean_str(login_info.get("token_user_id", ""))
    token_hub_id = _clean_str(login_info.get("token_hub_id", ""))
    token_app_id = _clean_str(login_info.get("token_app_id", ""))

    who = (
        token_user_email
        or (f"userId {token_user_id}" if token_user_id else "")
        or _clean_str(login_info.get("portal_name", ""))
        or f"Portal {_clean_str(login_info.get('portal_id', ''))}"
    )

    lines = [f"Connected as {who}."]

    if token_user_email:
        lines.append(f"login/email = {token_user_email}")
    if token_user_id:
        lines.append(f"userId = {token_user_id}")
    if token_hub_id:
        lines.append(f"hubId = {token_hub_id}")
    if token_app_id:
        lines.append(f"appId = {token_app_id}")

    owners_probe = probe_by_label.get("Owners", {}) or {}
    owners_ok = bool(owners_probe.get("ok"))
    owners_status = owners_probe.get("status_code", "")
    owners_results = _probe_payload_results(owners_probe.get("payload"))
    owners_preview = _build_people_preview(owners_results)

    if owners_ok:
        lines.append(f"Owners visible = {len(owners_results)}")
        if owners_preview:
            lines.append(f"owners: {owners_preview}")
    elif owners_status:
        lines.append(f"Owners API = no access ({owners_status})")

    users_probe = probe_by_label.get("Users", {}) or {}
    users_ok = bool(users_probe.get("ok"))
    users_status = users_probe.get("status_code", "")
    users_results = _probe_payload_results(users_probe.get("payload"))
    users_preview = _build_people_preview(users_results)

    if users_ok:
        lines.append(f"Users visible = {len(users_results)}")
        if users_preview:
            lines.append(f"users: {users_preview}")
    elif users_status:
        lines.append(f"Users API = no access ({users_status})")

    return "\n".join(lines)



def set_debug_visible(is_visible: bool) -> None:
    if "debug_shell" in globals():
        debug_shell.layout.display = "" if is_visible else "none"


def clear_debug_output() -> None:
    if "debug_output_area" in globals():
        with debug_output_area:
            clear_output()
    set_debug_visible(False)


# = = = = = =  = = = = = =  = = = = = =  = = = = = =  = = = = = =  = = = = = =  = = = = = = 

def get_token_user_owner_context(
    login_info: dict | None = None,
    companies_df: pd.DataFrame | None = None,
) -> dict:
    login_info = dict(login_info or STACK_UI_STATE.get("hubspot_login_info") or {})

    token_user_id = _clean_str(login_info.get("token_user_id", ""))
    token_hub_id = _clean_str(login_info.get("token_hub_id", ""))
    token_app_id = _clean_str(login_info.get("token_app_id", ""))
    token_user_email = _clean_str(login_info.get("token_user_email", ""))
    portal_id = _clean_str(login_info.get("portal_id", ""))
    portal_name = _clean_str(login_info.get("portal_name", ""))

    token_user_df = pd.DataFrame([{
        "token_user_id": token_user_id,
        "token_hub_id": token_hub_id,
        "token_app_id": token_app_id,
        "token_user_email": token_user_email,
        "portal_id": portal_id,
        "portal_name": portal_name,
    }])

    if companies_df is None:
        companies_df = STACK_UI_STATE.get("companies_df")
        if (
            (companies_df is None or not isinstance(companies_df, pd.DataFrame) or companies_df.empty)
            and STACK_UI_STATE.get("logged_in")
        ):
            try:
                companies_df = get_my_companies()
            except Exception:
                companies_df = pd.DataFrame()

    if not isinstance(companies_df, pd.DataFrame):
        companies_df = pd.DataFrame()

    company_owner_df = _ensure_columns(
        companies_df.copy(),
        ["id", "name", "domain", "hubspot_owner_id"],
    )

    if not company_owner_df.empty:
        company_owner_df["hubspot_owner_id"] = (
            company_owner_df["hubspot_owner_id"]
            .fillna("")
            .astype(str)
            .replace({"nan": "", "None": ""})
            .str.strip()
        )
        company_owner_df["owner_id_equals_token_user_id"] = (
            company_owner_df["hubspot_owner_id"].eq(token_user_id)
            if token_user_id else False
        )
    else:
        company_owner_df["owner_id_equals_token_user_id"] = pd.Series(dtype=bool)

    unique_owner_ids = sorted(
        {x for x in company_owner_df.get("hubspot_owner_id", pd.Series(dtype=str)).tolist() if _clean_str(x)}
    )

    unique_owner_ids_df = pd.DataFrame({"hubspot_owner_id": unique_owner_ids})

    summary_df = pd.DataFrame([{
        "token_user_id": token_user_id,
        "token_hub_id": token_hub_id,
        "token_app_id": token_app_id,
        "companies_found": int(len(company_owner_df)),
        "companies_with_owner_id": int(company_owner_df["hubspot_owner_id"].ne("").sum()) if not company_owner_df.empty else 0,
        "unique_owner_ids_found": int(len(unique_owner_ids)),
        "any_owner_id_equals_token_user_id": bool(token_user_id and token_user_id in unique_owner_ids),
    }])

    note_df = pd.DataFrame([{
        "note": (
            "Raw comparison only. Canonical owner resolution requires Owners API access: "
            "token userId joins to owners.userId; company hubspot_owner_id joins to owners.id."
        )
    }])

    return {
        "token_user_df": token_user_df,
        "company_owner_df": company_owner_df,
        "unique_owner_ids_df": unique_owner_ids_df,
        "summary_df": summary_df,
        "note_df": note_df,
    }


def build_identity_debug_bundle(
    login_info: dict | None = None,
    companies_df: pd.DataFrame | None = None,
) -> dict:
    login_info = dict(login_info or STACK_UI_STATE.get("hubspot_login_info") or {})
    probes = run_login_probes()

    if companies_df is None:
        try:
            companies_df = get_my_companies()
        except Exception:
            companies_df = pd.DataFrame()

    probe_by_label = {str(p.get("label", "")): p for p in (probes or [])}

    acct_payload = dict((probe_by_label.get("Account Details") or {}).get("payload") or {})
    token_payload = dict((probe_by_label.get("Access Token Info") or {}).get("payload") or {})

    identity_rows = [
        {"source": "token_info", "field": "userId", "value": token_payload.get("userId") or login_info.get("token_user_id", "")},
        {"source": "token_info", "field": "hubId", "value": token_payload.get("hubId") or login_info.get("token_hub_id", "")},
        {"source": "token_info", "field": "appId", "value": token_payload.get("appId") or login_info.get("token_app_id", "")},
        {"source": "token_info", "field": "isUserToken", "value": token_payload.get("isUserToken", "")},
        {"source": "login_info", "field": "token_user_email", "value": login_info.get("token_user_email", "")},
        {"source": "account_details", "field": "portalId", "value": acct_payload.get("portalId") or login_info.get("portal_id", "")},
        {"source": "account_details", "field": "accountType", "value": acct_payload.get("accountType", "")},
        {"source": "account_details", "field": "timeZone", "value": acct_payload.get("timeZone") or login_info.get("portal_timezone", "")},
        {"source": "account_details", "field": "companyCurrency", "value": acct_payload.get("companyCurrency") or login_info.get("portal_currency", "")},
    ]
    identity_rows = [row for row in identity_rows if row.get("value") not in ("", None, [], {})]
    identity_df = pd.DataFrame(identity_rows)

    scope_list = token_payload.get("scopes") or login_info.get("token_scopes", []) or []
    scope_df = pd.DataFrame({"scope": scope_list}) if scope_list else pd.DataFrame(columns=["scope"])

    probe_summary_df = pd.DataFrame([
        {
            "label": p.get("label", ""),
            "status_code": p.get("status_code", ""),
            "ok": p.get("ok", False),
            "url": p.get("url", ""),
        }
        for p in probes
    ])

    token_owner_ctx = get_token_user_owner_context(
        login_info=login_info,
        companies_df=companies_df,
    )

    return {
        "login_info": login_info,
        "probes": probes,
        "identity_df": identity_df,
        "scope_df": scope_df,
        "probe_summary_df": probe_summary_df,
        "token_owner_ctx": token_owner_ctx,
        "companies_df": companies_df,
    }


def render_identity_debug(bundle: dict) -> None:
    if "debug_output_area" not in globals():
        return

    probes = bundle.get("probes", []) or []
    identity_df = bundle.get("identity_df", pd.DataFrame())
    scope_df = bundle.get("scope_df", pd.DataFrame())
    probe_summary_df = bundle.get("probe_summary_df", pd.DataFrame())
    token_owner_ctx = bundle.get("token_owner_ctx", {}) or {}

    set_debug_visible(True)

    with debug_output_area:
        clear_output()

        display(HTML("<h4 style='margin:0 0 10px 0;'>DEBUG</h4>"))

        display(HTML("<div style='margin:0 0 8px 0; font-weight:600;'>Identity Attributes</div>"))
        display(identity_df)

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Token User / Company Owner Summary</div>"))
        display(token_owner_ctx.get("summary_df", pd.DataFrame()))

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Token User Detail</div>"))
        display(token_owner_ctx.get("token_user_df", pd.DataFrame()))

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Companies Found — Owner IDs</div>"))
        display(token_owner_ctx.get("company_owner_df", pd.DataFrame()))

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Unique Company Owner IDs</div>"))
        display(token_owner_ctx.get("unique_owner_ids_df", pd.DataFrame()))

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Comparison Note</div>"))
        display(token_owner_ctx.get("note_df", pd.DataFrame()))

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Token Scopes</div>"))
        display(scope_df)

        display(HTML("<div style='margin:14px 0 8px 0; font-weight:600;'>Probe Summary</div>"))
        display(probe_summary_df)

        for p in probes:
            label = p.get("label", "")
            status_code = p.get("status_code", "")
            ok = p.get("ok", False)
            payload = p.get("payload", {})

            display(HTML(
                f"<div style='margin:14px 0 6px 0; font-weight:600;'>"
                f"{_esc(label)} — status {status_code} — ok={ok}"
                f"</div>"
            ))
            display(_payload_preview_df(payload))


def handle_identity_debug(login_info: dict | None = None, companies_df: pd.DataFrame | None = None) -> None:
    """
    Single entry point for CB6 so the debug toggle logic is not duplicated.
    """
    if DEBUG_IDENTITY:
        bundle = build_identity_debug_bundle(
            login_info=login_info,
            companies_df=companies_df,
        )
        render_identity_debug(bundle)
    else:
        clear_debug_output()

print("CB4 done: Login / Probes / DEBUG Helpers")

CB4 done: Login / Probes / DEBUG Helpers


### CB5: Current User / Ownership Helpers

In [125]:
# =========================
# CB5 — Current User / Ownership Helpers
# =========================

def get_current_token_user_id() -> str:
    return _clean_str(
        STACK_UI_STATE.get("token_user_id")
        or (STACK_UI_STATE.get("hubspot_login_info") or {}).get("token_user_id")
        or ""
    )


def row_belongs_to_current_user(row, owner_col: str = OWNER_ID_COL) -> bool:
    token_user_id = get_current_token_user_id()
    owner_id = _clean_str(row.get(owner_col, ""))
    return bool(token_user_id) and owner_id == token_user_id


def object_dict_belongs_to_current_user(
    obj_dict: dict,
    owner_col: str = OWNER_ID_COL,
) -> bool:
    return row_belongs_to_current_user(obj_dict, owner_col=owner_col)


def filter_df_to_current_user(
    df: pd.DataFrame | None,
    *,
    owner_col: str = OWNER_ID_COL,
    out_cols: list[str] | None = None,
) -> pd.DataFrame:
    if df is None or not isinstance(df, pd.DataFrame):
        df = pd.DataFrame()

    if out_cols:
        df = _ensure_columns(df.copy(), out_cols)
    else:
        df = df.copy()

    if df.empty:
        return df

    if owner_col not in df.columns:
        df[owner_col] = ""

    df[owner_col] = (
        df[owner_col]
        .fillna("")
        .astype(str)
        .replace({"nan": "", "None": ""})
        .str.strip()
    )

    token_user_id = get_current_token_user_id()
    if not token_user_id:
        return df.iloc[0:0].copy()

    return df[df[owner_col] == token_user_id].copy()

print("CB5 done: Current User / Ownership Helpers")

CB5 done: Current User / Ownership Helpers


### CB6: Data Functions

In [126]:
# =========================
# CB2 — Data Functions
# =========================
# =========================
# CB2 — Data Functions
# =========================

COMPANY_PROPS = [
    "hs_object_id",
    "name",
    "domain",
    "website",
    "phone",
    "city",
    "state",
    "country",
    "createdate",
    "hs_lastmodifieddate",
    "hubspot_owner_id",
]

MEETING_PROPS = [
    "id",
    "hs_body_preview",
    "hs_createdate",
    "hs_object_id",
    "hs_meeting_start_time",
    "hs_meeting_end_time",
    "hubspot_owner_id",
]

CONTACT_PROPS = [
    "firstname",
    "lastname",
    "email",
    "phone",
    "createdate",
    "hs_object_id",
    "hubspot_owner_id",
]

OWNED_COMPANY_OUT_COLS = [
    "id",
    "name",
    "domain",
    "website",
    "phone",
    "city",
    "state",
    "country",
    "hubspot_owner_id",
]


# ---------------------------------------------------------
# Generic object helpers
# ---------------------------------------------------------
def _objects_to_df(objects, out_cols: list[str]) -> pd.DataFrame:
    df = pd.DataFrame(
        [{"id": obj.id, **(obj.properties or {})} for obj in (objects or [])]
    )
    return _ensure_columns(df, out_cols)


def _fetch_objects_by_id(
    fetch_fn,
    ids: list[str],
    properties: list[str],
    out_cols: list[str] | None = None,
) -> pd.DataFrame:
    rows = []
    for obj_id in ids:
        obj = fetch_fn(obj_id, properties=properties)
        rows.append({"id": obj.id, **(obj.properties or {})})

    out_cols = out_cols or (["id"] + properties)
    return _ensure_columns(pd.DataFrame(rows), out_cols)


def get_associated_ids(
    from_type: str,
    from_id: str,
    to_type: str,
    limit: int = 100,
) -> list[str]:
    _require_login()

    r = api.crm.associations.v4.basic_api.get_page(
        object_type=from_type,
        object_id=str(from_id),
        to_object_type=to_type,
        limit=limit,
    )
    return [str(x.to_object_id) for x in (r.results or [])]


def _format_contact_label(row: dict) -> str:
    first = _clean_str(row.get("firstname", ""))
    last = _clean_str(row.get("lastname", ""))
    email = _clean_str(row.get("email", ""))

    name = " ".join(x for x in [first, last] if x).strip()
    if name and email:
        return f"{name} <{email}>"
    if name:
        return name
    return email


# ---------------------------------------------------------
# Company functions
# ---------------------------------------------------------
def get_my_companies(limit: int = 100) -> pd.DataFrame:
    _require_login()

    page = api.crm.companies.basic_api.get_page(
        limit=limit,
        properties=COMPANY_PROPS,
        archived=False,
    )

    df_companies = _objects_to_df(page.results, ["id"] + COMPANY_PROPS)
    STACK_UI_STATE["companies_df"] = df_companies
    return df_companies


def get_my_owned_companies_df(df_companies: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Single canonical owned-company filter.
    """
    if df_companies is None or not isinstance(df_companies, pd.DataFrame):
        df_companies = STACK_UI_STATE.get("companies_df")

    if df_companies is None or not isinstance(df_companies, pd.DataFrame) or df_companies.empty:
        df_companies = get_my_companies()

    df = filter_df_to_current_user(
        df_companies,
        owner_col="hubspot_owner_id",
        out_cols=OWNED_COMPANY_OUT_COLS,
    )

    if df.empty:
        df = _ensure_columns(df, OWNED_COMPANY_OUT_COLS)
        df["homepage_url"] = pd.Series(dtype=str)
    else:
        df["homepage_url"] = df.apply(get_company_homepage_url, axis=1)

    STACK_UI_STATE["companies_display_df"] = df
    STACK_UI_STATE["my_owned_companies_df"] = df

    return df


def _build_company_browser_rows(df_display: pd.DataFrame, browser_html: widgets.HTML) -> list:
    row_widgets = []

    for _, row in df_display.iterrows():
        company_id = _clean_str(row.get("id", ""))
        company_name = (
            _clean_str(row.get("name", ""))
            or _clean_str(row.get("domain", ""))
            or company_id
        )
        domain = _clean_str(row.get("domain", ""))
        website = _clean_str(row.get("website", ""))
        owner_id = _clean_str(row.get("hubspot_owner_id", ""))
        url = _clean_str(row.get("homepage_url", ""))

        open_btn = widgets.Button(
            description="Open",
            layout=widgets.Layout(width="80px", height="32px"),
        )

        row_html = widgets.HTML(
            value=(
                f"<div style='line-height:1.35;'>"
                f"<div><b>{_esc(company_name)}</b></div>"
                f"<div>ID: {_esc(company_id)}</div>"
                f"<div>Domain: {_esc(domain)}</div>"
                f"<div>Website: {_esc(website)}</div>"
                f"<div>Owner ID: {_esc(owner_id)}</div>"
                f"</div>"
            ),
            layout=widgets.Layout(width="100%"),
        )

        def _make_handler(url=url, company_name=company_name):
            def _handler(_):
                if url:
                    browser_html.value = (
                        f"<div style='font-weight:600; margin:0 0 8px 0;'>"
                        f"{_esc(company_name)}"
                        f"</div>"
                        f"<div style='margin:0 0 8px 0;'>"
                        f"<a href='{_esc(url)}' target='_blank' rel='noopener noreferrer'>"
                        f"{_esc(url)}"
                        f"</a>"
                        f"</div>"
                        f"<iframe "
                        f"src='{_esc(url)}' "
                        f"style='width:100%; height:520px; border:1px solid #d8d8d8; "
                        f"border-radius:8px; background:white;'>"
                        f"</iframe>"
                    )
                else:
                    browser_html.value = (
                        f"<div style='color:#666; padding:10px 0;'>"
                        f"No homepage URL available for {_esc(company_name)}."
                        f"</div>"
                    )
            return _handler

        open_btn.on_click(_make_handler())

        row_widgets.append(
            widgets.HBox(
                [open_btn, row_html],
                layout=widgets.Layout(
                    width="100%",
                    align_items="center",
                    gap="10px",
                ),
            )
        )

    return row_widgets


def render_my_companies_output(df_owned_companies: pd.DataFrame) -> None:
    """
    Render already-filtered owned companies only.
    """
    if "output_area" not in globals():
        return

    df_display = _ensure_columns(
        df_owned_companies,
        OWNED_COMPANY_OUT_COLS + ["homepage_url"],
    )

    token_user_id = get_current_token_user_id()

    browser_html = widgets.HTML(
        value=(
            "<div style='color:#666; padding:10px 0;'>"
            "Click Open to load the selected company site below."
            "</div>"
        ),
        layout=widgets.Layout(width="100%"),
    )

    row_widgets = _build_company_browser_rows(df_display, browser_html)

    with output_area:
        clear_output()

        display(HTML("<h4 style='margin:0 0 10px 0;'>My Companies</h4>"))

        display(HTML(
            "<div style='margin:0 0 8px 0; font-weight:600;'>"
            "Company Current News"
            "</div>"
        ))

        if row_widgets:
            display(
                widgets.VBox(
                    row_widgets,
                    layout=widgets.Layout(width="100%", gap="8px"),
                )
            )
        else:
            display(HTML(
                f"<div style='color:#666; padding:10px 0;'>"
                f"No companies found where hubspot_owner_id matches token user id {_esc(token_user_id)}."
                f"</div>"
            ))

        display(HTML(
            "<div style='margin:18px 0 8px 0; font-weight:600;'>"
            "Selected Company Site"
            "</div>"
        ))
        display(browser_html)


# ---------------------------------------------------------
# Meeting / contact detail helpers
# ---------------------------------------------------------
def get_company_contacts_for_companies(companies_df: pd.DataFrame):
    """
    For each associated company, pull its associated contacts,
    then keep only contacts owned by the current login user.
    """
    if companies_df is None or companies_df.empty:
        out_cols = [
            "source_company_id",
            "source_company_name",
            "id",
            "firstname",
            "lastname",
            "email",
            "phone",
            "createdate",
            "hs_object_id",
            "hubspot_owner_id",
        ]
        return pd.DataFrame(columns=out_cols), ""

    contact_props = [
        "firstname",
        "lastname",
        "email",
        "phone",
        "createdate",
        "hs_object_id",
        "hubspot_owner_id",
    ]

    rows = []
    summary_parts = []

    for _, crow in companies_df.iterrows():
        company_id = _clean_str(crow.get("id", ""))
        company_name = _clean_str(crow.get("name", "")) or company_id

        if not company_id:
            continue

        try:
            company_contact_ids = get_associated_ids("companies", company_id, "contacts")
        except Exception:
            company_contact_ids = []

        if not company_contact_ids:
            continue

        company_contacts_df = _fetch_objects_by_id(
            api.crm.contacts.basic_api.get_by_id,
            company_contact_ids,
            contact_props,
        )

        company_contacts_df = filter_df_to_current_user(
            company_contacts_df,
            owner_col="hubspot_owner_id",
            out_cols=["id"] + contact_props,
        )

        if company_contacts_df.empty:
            continue

        company_contacts_df["source_company_id"] = company_id
        company_contacts_df["source_company_name"] = company_name

        labels = [
            _format_contact_label(rec)
            for rec in company_contacts_df.to_dict("records")
            if _format_contact_label(rec)
        ]
        if labels:
            summary_parts.append(f"{company_name}: " + ", ".join(labels))

        rows.extend(company_contacts_df.to_dict("records"))

    out_cols = [
        "source_company_id",
        "source_company_name",
        "id",
        "firstname",
        "lastname",
        "email",
        "phone",
        "createdate",
        "hs_object_id",
        "hubspot_owner_id",
    ]

    company_contacts_df = _ensure_columns(pd.DataFrame(rows), out_cols)
    company_contacts_summary = " | ".join(summary_parts)

    return company_contacts_df, company_contacts_summary


def get_meeting_association_details(meeting_id: str):
    _require_login()

    meeting_id = str(meeting_id).strip()
    if not meeting_id:
        raise RuntimeError("meeting_id is required.")

    company_ids = get_associated_ids("meetings", meeting_id, "companies")
    contact_ids = get_associated_ids("meetings", meeting_id, "contacts")
    deal_ids = get_associated_ids("meetings", meeting_id, "deals")
    ticket_ids = get_associated_ids("meetings", meeting_id, "tickets")

    company_props = [
        "name",
        "domain",
        "website",
        "createdate",
        "hs_lastmodifieddate",
        "hs_object_id",
        "hubspot_owner_id",
    ]
    contact_props = [
        "firstname",
        "lastname",
        "email",
        "phone",
        "createdate",
        "hs_object_id",
        "hubspot_owner_id",
    ]
    deal_props = [
        "dealname",
        "dealstage",
        "amount",
        "createdate",
        "hs_lastmodifieddate",
        "hs_object_id",
        "hubspot_owner_id",
    ]
    ticket_props = [
        "subject",
        "content",
        "hs_pipeline_stage",
        "hs_object_id",
    ]

    companies_df = _fetch_objects_by_id(
        api.crm.companies.basic_api.get_by_id,
        company_ids,
        company_props,
    )
    contacts_df = _fetch_objects_by_id(
        api.crm.contacts.basic_api.get_by_id,
        contact_ids,
        contact_props,
    )
    deals_df = _fetch_objects_by_id(
        api.crm.deals.basic_api.get_by_id,
        deal_ids,
        deal_props,
    )
    tickets_df = _fetch_objects_by_id(
        api.crm.tickets.basic_api.get_by_id,
        ticket_ids,
        ticket_props,
    )

    companies_df = filter_df_to_current_user(
        companies_df,
        owner_col="hubspot_owner_id",
        out_cols=["id"] + company_props,
    )
    contacts_df = filter_df_to_current_user(
        contacts_df,
        owner_col="hubspot_owner_id",
        out_cols=["id"] + contact_props,
    )
    deals_df = filter_df_to_current_user(
        deals_df,
        owner_col="hubspot_owner_id",
        out_cols=["id"] + deal_props,
    )

    company_contacts_df, company_contacts_summary = get_company_contacts_for_companies(companies_df)

    company_ids = [
        str(x).strip()
        for x in companies_df.get("id", pd.Series(dtype=str)).tolist()
        if str(x).strip()
    ]
    contact_ids = [
        str(x).strip()
        for x in contacts_df.get("id", pd.Series(dtype=str)).tolist()
        if str(x).strip()
    ]
    deal_ids = [
        str(x).strip()
        for x in deals_df.get("id", pd.Series(dtype=str)).tolist()
        if str(x).strip()
    ]
    ticket_ids = [
        str(x).strip()
        for x in tickets_df.get("id", pd.Series(dtype=str)).tolist()
        if str(x).strip()
    ]

    return {
        "meeting_id": meeting_id,
        "company_ids": company_ids,
        "contact_ids": contact_ids,
        "deal_ids": deal_ids,
        "ticket_ids": ticket_ids,
        "companies_df": companies_df,
        "contacts_df": contacts_df,
        "deals_df": deals_df,
        "tickets_df": tickets_df,
        "company_contacts_df": company_contacts_df,
        "company_contacts_summary": company_contacts_summary,
    }


def get_token_user_company_ids(limit: int = 100) -> list[str]:
    """
    Return company ids owned by the current token user.
    Reuse cached owned companies when available.
    """
    _require_login()

    df = STACK_UI_STATE.get("my_owned_companies_df")
    if df is None or not isinstance(df, pd.DataFrame):
        df = get_my_owned_companies_df(get_my_companies(limit=limit))

    if df.empty or "id" not in df.columns:
        return []

    return [str(x).strip() for x in df["id"].tolist() if str(x).strip()]


def get_meetings_for_token_user(limit: int = 100) -> pd.DataFrame:
    """
    Return only meetings owned by the current login user and associated
    to companies also owned by the current login user.
    Include readable associated company/contact/deal columns.
    """
    _require_login()

    token_user_id = get_current_token_user_id()
    out_cols = [
        "id",
        "hubspot_owner_id",
        "hs_meeting_start_time",
        "hs_meeting_end_time",
        "hs_createdate",
        "hs_object_id",
        "hs_body_preview",
        "associated_company_ids",
        "associated_company_names",
        "associated_contact_ids",
        "associated_contact_names",
        "associated_contact_emails",
        "associated_deal_ids",
        "associated_deal_names",
        "company_contacts_summary",
        "matched_company_ids",
        "token_user_id",
    ]

    if not token_user_id:
        return pd.DataFrame(columns=out_cols)

    owned_company_ids = set(get_token_user_company_ids(limit=limit))
    if not owned_company_ids:
        return pd.DataFrame(columns=out_cols)

    page = api.crm.objects.basic_api.get_page(
        object_type="meetings",
        limit=limit,
        properties=[c for c in MEETING_PROPS if c != "id"],
        archived=False,
    )

    rows = []
    for m in page.results:
        row = {"id": str(m.id), **(m.properties or {})}

        if not object_dict_belongs_to_current_user(row, owner_col="hubspot_owner_id"):
            continue

        try:
            assoc = get_meeting_association_details(row["id"])
        except Exception:
            continue

        assoc_company_ids = assoc["company_ids"]
        matched_company_ids = sorted(owned_company_ids.intersection(assoc_company_ids))
        if not matched_company_ids:
            continue

        companies_df = assoc["companies_df"]
        contacts_df = assoc["contacts_df"]
        deals_df = assoc["deals_df"]

        company_names = []
        if not companies_df.empty:
            company_names = [
                _clean_str(x)
                for x in companies_df.get("name", pd.Series(dtype=str)).tolist()
                if _clean_str(x)
            ]

        contact_names = []
        contact_emails = []
        if not contacts_df.empty:
            contact_names = [
                _format_contact_label(rec)
                for rec in contacts_df.to_dict("records")
                if _format_contact_label(rec)
            ]
            contact_emails = [
                _clean_str(x)
                for x in contacts_df.get("email", pd.Series(dtype=str)).tolist()
                if _clean_str(x)
            ]

        deal_names = []
        if not deals_df.empty:
            deal_names = [
                _clean_str(x)
                for x in deals_df.get("dealname", pd.Series(dtype=str)).tolist()
                if _clean_str(x)
            ]

        row["associated_company_ids"] = ", ".join(assoc_company_ids)
        row["associated_company_names"] = ", ".join(company_names)
        row["associated_contact_ids"] = ", ".join(assoc["contact_ids"])
        row["associated_contact_names"] = ", ".join(contact_names)
        row["associated_contact_emails"] = ", ".join(contact_emails)
        row["associated_deal_ids"] = ", ".join(assoc["deal_ids"])
        row["associated_deal_names"] = ", ".join(deal_names)
        row["company_contacts_summary"] = assoc.get("company_contacts_summary", "")
        row["matched_company_ids"] = ", ".join(matched_company_ids)
        row["token_user_id"] = token_user_id
        rows.append(row)

    df_meetings = _ensure_columns(pd.DataFrame(rows), out_cols)
    STACK_UI_STATE["meetings_df"] = df_meetings
    STACK_UI_STATE["meeting_company_filter_mode"] = "token_user_owned"
    STACK_UI_STATE["meeting_company_filter_value"] = token_user_id

    return df_meetings

# def _build_meeting_browser_rows(df_display: pd.DataFrame, detail_html: widgets.HTML) -> list:
def _build_meeting_browser_rows(df_display: pd.DataFrame, detail_html: widgets.HTML) -> list:
    row_widgets = []

    for _, row in df_display.iterrows():
        meeting_id = _clean_str(row.get("id", ""))
        company_contacts_summary = _clean_str(row.get("company_contacts_summary", ""))
        deal_names = _clean_str(row.get("associated_deal_names", ""))
        deal_ids = _clean_str(row.get("associated_deal_ids", ""))
        start_time = _clean_str(row.get("hs_meeting_start_time", ""))
        end_time = _clean_str(row.get("hs_meeting_end_time", ""))
        body_preview = _clean_str(row.get("hs_body_preview", ""))

        open_btn = widgets.Button(
            description="Open",
            layout=widgets.Layout(width="80px", height="32px"),
        )

        row_title = company_contacts_summary or f"Meeting {meeting_id}"

        row_html = widgets.HTML(
            value=(
                f"<div style='line-height:1.35;'>"
                f"<div><b>{_esc(row_title)}</b></div>"
                f"<div>Start: {_esc(start_time)}</div>"
                f"</div>"
            ),
            layout=widgets.Layout(width="100%"),
        )

        def _make_handler(
            row_title=row_title,
            start_time=start_time,
            end_time=end_time,
            deal_names=deal_names,
            deal_ids=deal_ids,
            body_preview=body_preview,
        ):
            def _handler(_):
                deal_line = ""
                if deal_names and deal_ids:
                    deal_line = f"<div style='margin:0 0 6px 0;'><b>Deals/IDs:</b> {_esc(deal_names)} (deal id={_esc(deal_ids)})</div>"
                elif deal_names:
                    deal_line = f"<div style='margin:0 0 6px 0;'><b>Deals/IDs:</b> {_esc(deal_names)}</div>"
                elif deal_ids:
                    deal_line = f"<div style='margin:0 0 6px 0;'><b>Deals/IDs:</b> deal id={_esc(deal_ids)}</div>"

                detail_html.value = (
                    f"<div style='font-weight:600; margin:0 0 8px 0;'>{_esc(row_title)}</div>"
                    f"<div style='margin:0 0 6px 0;'><b>Date/Time:</b> {_esc(start_time)} - {_esc(end_time)}</div>"
                    f"{deal_line}"
                    f"<div style='margin:10px 0 6px 0;'><b>Meeting Notes / Preview:</b></div>"
                    f"<div style='white-space:pre-wrap; border:1px solid #d8d8d8; border-radius:8px; "
                    f"padding:10px; background:white;'>{_esc(body_preview)}</div>"
                )
            return _handler

        open_btn.on_click(_make_handler())

        row_widgets.append(
            widgets.HBox(
                [open_btn, row_html],
                layout=widgets.Layout(
                    width="100%",
                    align_items="center",
                    gap="10px",
                ),
            )
        )

    return row_widgets









def render_meeting_prep_output(df_meetings: pd.DataFrame) -> None:
    if "output_area" not in globals():
        return

    display_cols = [
        "id",
        "company_contacts_summary",
        "associated_deal_names",
        "associated_deal_ids",
        "hs_meeting_start_time",
        "hs_meeting_end_time",
        "hs_body_preview",
    ]

    df_display = _ensure_columns(df_meetings, display_cols)

    detail_html = widgets.HTML(
        value=(
            "<div style='color:#666; padding:10px 0;'>"
            "Click Open to load meeting details below."
            "</div>"
        ),
        layout=widgets.Layout(width="100%"),
    )

    row_widgets = _build_meeting_browser_rows(df_display, detail_html)

    with output_area:
        clear_output()

        display(HTML("<h4 style='margin:0 0 10px 0;'>Meeting Prep</h4>"))

        display(HTML(
            "<div style='margin:0 0 8px 0; font-weight:600;'>"
            "My Meetings"
            "</div>"
        ))

        if row_widgets:
            display(
                widgets.VBox(
                    row_widgets,
                    layout=widgets.Layout(width="100%", gap="8px"),
                )
            )
        else:
            display(HTML(
                "<div style='color:#666; padding:10px 0;'>"
                "No meetings found for the current login user."
                "</div>"
            ))

        display(HTML(
            "<div style='margin:18px 0 8px 0; font-weight:600;'>"
            "Selected Meeting Details"
            "</div>"
        ))
        display(detail_html)


def get_meeting_prep(limit: int = 100) -> pd.DataFrame:
    return get_meetings_for_token_user(limit=limit)


print("CB6- Data Functions done")

CB6- Data Functions done


###  CB10: UI Layout / Widgets

In [127]:
# =========================
# CB10 — STACK UI Layout / Widgets
# =========================

# ---------------------------------------------------------
# Layout constants
# ---------------------------------------------------------
LEFT_W = "28%"
RIGHT_W = "72%"


# ---------------------------------------------------------
# Header row with login / logout buttons
# ---------------------------------------------------------
login_btn = widgets.Button(
    description="Log In",
    layout=widgets.Layout(width="100px", height="34px"),
)
login_btn.add_class("stack-login-btn")

logout_btn = widgets.Button(
    description="Logout",
    button_style="warning",
    layout=widgets.Layout(width="100px", height="34px"),
)
logout_btn.disabled = not STACK_UI_STATE.get("logged_in", False)

header_left_spacer = widgets.Box(
    layout=widgets.Layout(width="100px")
)

header_title = widgets.HTML(
    value="<div class='stack-header-title'>Stack</div>",
    layout=widgets.Layout(width="100%"),
)

header_right = widgets.HBox(
    [login_btn, logout_btn],
    layout=widgets.Layout(
        width="220px",
        justify_content="flex-end",
        align_items="center",
        gap="10px",
    ),
)

title_bar = widgets.GridBox(
    [header_left_spacer, header_title, header_right],
    layout=widgets.Layout(
        width="100%",
        grid_template_columns="100px 1fr 220px",
        align_items="center",
    ),
)
title_bar.add_class("stack-header-row")


# ---------------------------------------------------------
# Left panel: Stack Agent + action buttons
# ---------------------------------------------------------
agent_card = widgets.HTML(
    value=(
        "<div class='stack-agent-card'>"
        "  <div class='stack-agent-title'>Stack Agent</div>"
        f"  <div class='stack-agent-text'>{_esc(STACK_UI_STATE['agent_text'])}</div>"
        "</div>"
    ),
    layout=widgets.Layout(width="100%"),
)

opp_insights_btn = widgets.Button(
    description="Opp Insights (not ready)",
    layout=widgets.Layout(width="100%", height="78px"),
)

meeting_prep_btn = widgets.Button(
    description="Meeting Prep",
    layout=widgets.Layout(width="100%", height="78px"),
)

my_companies_btn = widgets.Button(
    description="My Companies",
    layout=widgets.Layout(width="100%", height="78px"),
)

opp_search_btn = widgets.Button(
    description="Opp Search (not ready)",
    layout=widgets.Layout(width="100%", height="78px"),
)

for _btn in [opp_insights_btn, meeting_prep_btn, my_companies_btn, opp_search_btn]:
    _btn.add_class("stack-main-btn")

left_col = widgets.VBox(
    [
        agent_card,
        widgets.HTML("<div style='height:18px;'></div>"),
        opp_insights_btn,
        widgets.HTML("<div style='height:14px;'></div>"),
        meeting_prep_btn,
        widgets.HTML("<div style='height:14px;'></div>"),
        my_companies_btn,
        widgets.HTML("<div style='height:14px;'></div>"),
        opp_search_btn,
    ],
    layout=widgets.Layout(
        width=LEFT_W,
        align_items="stretch",
    ),
)


# ---------------------------------------------------------
# Right panel: output frame
# ---------------------------------------------------------
output_title = widgets.HTML(
    value="<div class='stack-panel-title'>Output</div>"
)

output_area = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        min_height="280px",
    )
)

right_panel = widgets.VBox(
    [
        output_title,
        output_area,
    ],
    layout=widgets.Layout(width="100%"),
)
right_panel.add_class("stack-panel")

right_col = widgets.VBox(
    [right_panel],
    layout=widgets.Layout(width=RIGHT_W),
)


# ---------------------------------------------------------
# Main body
# ---------------------------------------------------------
main_row = widgets.HBox(
    [left_col, right_col],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start",
        justify_content="space-between",
        gap="20px",
    ),
)


# ---------------------------------------------------------
# Bottom DEBUG panel (hidden by default)
# ---------------------------------------------------------
debug_title = widgets.HTML(
    value="<div class='stack-panel-title'>DEBUG</div>"
)

debug_output_area = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        min_height="240px",
    )
)

debug_panel = widgets.VBox(
    [
        debug_title,
        debug_output_area,
    ],
    layout=widgets.Layout(width="100%"),
)
debug_panel.add_class("stack-panel")

debug_shell = widgets.VBox(
    [debug_panel],
    layout=widgets.Layout(
        width="100%",
        display="none",
        margin="18px 0 0 0",
    ),
)


# ---------------------------------------------------------
# App root
# ---------------------------------------------------------
ui = widgets.VBox(
    [title_bar, main_row, debug_shell],
    layout=widgets.Layout(width="100%"),
)
ui.add_class("stack-root")


# ---------------------------------------------------------
# Initial content
# ---------------------------------------------------------
show_output(
    "Output",
    "Click Log In or one of the action buttons to display results here."
)
clear_debug_output()

print("CB10 UI Layout Widgets done")

CB10 UI Layout Widgets done


### CB11: UI Login Widget Sync

In [128]:
# =========================
# CB11 — UI Login Widget Sync
# =========================

def sync_login_widgets():
    logged_in = bool(STACK_UI_STATE.get("logged_in"))

    if "login_btn" in globals():
        login_btn.description = "Connected" if logged_in else "Log In"
        login_btn.disabled = logged_in

    if "logout_btn" in globals():
        logout_btn.disabled = not logged_in


sync_login_widgets()

print("CB5- UI Login widget sync done")

CB5- UI Login widget sync done


### CB12: STACK UI Button Actions / Callbacks

In [129]:
# =========================
# CB12 — STACK UI Button Actions / Callbacks
# =========================

def on_login_clicked(_):
    if STACK_UI_STATE.get("logged_in"):
        login_info = dict(STACK_UI_STATE.get("hubspot_login_info") or {})
        STACK_UI_STATE["last_action"] = "login"
        sync_login_widgets()

        set_agent_text(build_connected_agent_text(login_info))
        show_output("Log In", build_login_display_df(login_info))
        handle_userid_debug(login_info=login_info)
        return

    set_agent_text("Connecting to HubSpot...")

    try:
        login_info = run_hubspot_login()
        STACK_UI_STATE["last_action"] = "login"
        sync_login_widgets()

        set_agent_text(build_connected_agent_text(login_info))
        show_output("Log In", build_login_display_df(login_info))
        handle_userid_debug(login_info=login_info)

    except Exception as e:
        reset_hubspot_login_state()
        sync_login_widgets()
        clear_debug_output()
        set_agent_text("HubSpot connection failed.")
        show_output("Log In", f"Login failed:\n\n{e}")


def on_logout_clicked(_):
    reset_hubspot_login_state()
    sync_login_widgets()
    clear_debug_output()
    set_agent_text("Logged out of HubSpot. Click Log In to test again.")
    show_output("Logout", "HubSpot session cleared.")


def on_opp_insights_clicked(_):
    STACK_UI_STATE["last_action"] = "opp_insights"
    set_agent_text("Running Opportunity Insights...")

    try:
        result = get_opp_insights()
        show_output("Opp Insights", result)
        set_agent_text("Opportunity Insights complete.")
    except Exception as e:
        show_output("Opp Insights", f"Error:\n\n{e}")
        set_agent_text("Opportunity Insights failed.")


def on_meeting_prep_clicked(_):
    STACK_UI_STATE["last_action"] = "meeting_prep"
    token_user_id = get_current_token_user_id()

    set_agent_text(f"Preparing meeting summary for userId {token_user_id}...")

    try:
        result = get_meeting_prep()
        render_meeting_prep_output(result)
        set_agent_text(
            f"Meeting Prep complete for userId {token_user_id}: {len(result)} meeting(s)."
        )
    except Exception as e:
        show_output("Meeting Prep", f"Error:\n\n{e}")
        set_agent_text("Meeting Prep failed.")


        
def on_my_companies_clicked(_):
    STACK_UI_STATE["last_action"] = "my_companies"
    set_agent_text("Loading My Companies...")

    try:
        owned_df = get_my_owned_companies_df(get_my_companies())
        render_my_companies_output(owned_df)
        set_agent_text(f"My Companies loaded: {len(owned_df)} row(s).")
    except Exception as e:
        show_output("My Companies", f"Error:\n\n{e}")
        set_agent_text("My Companies failed.")



def on_opp_search_clicked(_):
    STACK_UI_STATE["last_action"] = "opp_search"
    set_agent_text("Searching opportunities...")

    try:
        result = get_opp_search()
        show_output("Opp Search (not ready)", result)
        set_agent_text("Opportunity search complete.")
    except Exception as e:
        show_output("Opp Search (not ready)", f"Error:\n\n{e}")
        set_agent_text("Opportunity search failed.")


# ---------------------------------------------------------
# Wire up buttons
# ---------------------------------------------------------
login_btn.on_click(on_login_clicked)
logout_btn.on_click(on_logout_clicked)
opp_insights_btn.on_click(on_opp_insights_clicked)
meeting_prep_btn.on_click(on_meeting_prep_clicked)
my_companies_btn.on_click(on_my_companies_clicked)
opp_search_btn.on_click(on_opp_search_clicked)

sync_login_widgets()
sync_action_buttons()

print("CB12- UI Button Actions / Callbacks done")

CB12- UI Button Actions / Callbacks done


### CBXX: Run UI

In [130]:
# =========================
# CBXX — Run UI
# =========================

show_output(
    "Output",
    "Click one of the center buttons to run a function and display results here."
)

display(ui)

print("✅ Stack UI ready")

✅ Stack UI ready
